In [0]:
%run ./survivor_common_business

In [0]:
def process_consumer_l2_data(consumer_join_sources_df, task_id, group_cols):
    media_df = spark.table(media_table)
    phone_df = spark.table(phone_table)
    address_df = spark.table(address_table)
    optin_df = spark.table(optin_table)

    #1.1 排除acs
    consumer_noacs_df = consumer_join_sources_df.filter(F.col("is_acs_source") != 1)

    #1.2 acs
    # consumer_acs_df = consumer_join_sources_df.filter(F.col("is_acs_source") == 1)
    consumer_acs_df = get_all_acs_consumer_data()

    #1.3 jpn（trakuten&老系统 置空name&birth信息）、twn（onlshell过滤）
    jpn_df = consumer_noacs_df.filter(F.col("scon_mrkt_code") == "JPN")
    twn_df = consumer_noacs_df.filter(F.col("scon_mrkt_code") == "TWN")
    others_df = consumer_noacs_df.filter(~F.col("scon_mrkt_code").isin(["JPN", "TWN"]))
    
    consumer_by_filter_df = others_df.unionByName(update_jpn_info(jpn_df, group_cols)).unionByName(filter_twn_data(twn_df, group_cols))
    consumer_by_filter_df.cache()
    print_log(f"consumer_by_filter_df count: {consumer_by_filter_df.count()}")


    # 其他都传排除acs的df，只有media、phone、address要传入包含acs的df
    # media、phone、address、opitn 需要排除trakuten profile
    bDerivedEmailOptin_df, media_json_df = generate_media_json(consumer_by_filter_df.filter(~(F.col("scon_srcs_code").isin(JPN_TRAKUTEN_LIST))), 
                                                               media_df, optin_df, consumer_acs_df, group_cols)
    bDerivedPhoneOptin_df, phone_json_df = generate_phone_json(consumer_by_filter_df.filter(~(F.col("scon_srcs_code").isin(JPN_TRAKUTEN_LIST))), 
                                                               phone_df, optin_df, consumer_acs_df, group_cols)
    bDerivedAddressOptin_df, address_json_df = generate_address_json(consumer_by_filter_df.filter(~(F.col("scon_srcs_code").isin(JPN_TRAKUTEN_LIST))), 
                                                                     address_df, optin_df, consumer_acs_df, group_cols)
    optin_json_df = generate_optin_json(consumer_by_filter_df.filter(~(F.col("scon_srcs_code").isin(JPN_TRAKUTEN_LIST))), 
                                        bDerivedEmailOptin_df, bDerivedPhoneOptin_df, bDerivedAddressOptin_df, optin_df, group_cols)

    source_system_df = generate_source_system_json(consumer_noacs_df, group_cols)
    hobby_df = generate_hobby_json(consumer_join_sources_df, hobby_table, group_cols)
    crossbrand_optin_df = generate_crossbrand_optin_json(consumer_join_sources_df, crossbrand_optin_table, group_cols)
    attributes_df = generate_attributes_json(consumer_join_sources_df, attributes_table, group_cols)
    auxiliary_df = generate_auxiliary_json(consumer_join_sources_df, auxiliary_table, group_cols)
    hair_type_df = generate_hair_type_json(consumer_join_sources_df, hair_type_table, group_cols)
    makeup_concerns_df = generate_makeup_concerns_json(consumer_join_sources_df, makeup_concerns_table, group_cols)
    hair_concern_df = generate_hair_concerns_json(consumer_join_sources_df, hair_concerns_table, group_cols)
    skin_concerns_df = generate_skin_concerns_json(consumer_join_sources_df, skin_concerns_table, group_cols)
    terms_df = generate_terms_json(consumer_join_sources_df, terms_table, group_cols)
    consumer_group_df = generate_consumer_group_json(consumer_join_sources_df, consumer_group_table, group_cols)
    remark_df = generate_remark_json(consumer_join_sources_df, remark_table, group_cols)
    notes_df = generate_notes_json(consumer_join_sources_df, notes_table, group_cols)

    # 计算consumer及各contact（5张表）的最大时间戳
    overall_maxtimestamp_df = calculate_overall_maxtimestamp(consumer_by_filter_df, media_df, phone_df, address_df, optin_df, group_cols)
    cbr_records_df = fetch_cbr_records(consumer_by_filter_df, overall_maxtimestamp_df, group_cols).withColumn("scvlevel", F.lit(2))
    
    result_df = cbr_records_df.join(source_system_df, group_cols, "left") \
        .join(hobby_df, group_cols, "left") \
        .join(media_json_df, group_cols, "left") \
        .join(phone_json_df, group_cols, "left") \
        .join(address_json_df, group_cols, "left") \
        .join(optin_json_df, group_cols, "left") \
        .join(crossbrand_optin_df, group_cols, "left") \
        .join(attributes_df, group_cols, "left") \
        .join(auxiliary_df, group_cols, "left") \
        .join(hair_type_df, group_cols, "left") \
        .join(makeup_concerns_df, group_cols, "left") \
        .join(hair_concern_df, group_cols, "left") \
        .join(skin_concerns_df, group_cols, "left") \
        .join(terms_df, group_cols, "left") \
        .join(consumer_group_df, group_cols, "left") \
        .join(remark_df, group_cols, "left") \
        .join(notes_df, group_cols, "left") \
        .withColumn("UPDATE_DT", F.current_timestamp()) \
        .withColumn("UPDATE_UID", F.lit(None)) \
        .withColumn("TASK_ID", F.lit(task_id)) \
        .withColumn("RecordTimeStamp", F.current_timestamp()) \
        .withColumn("RecordUUID", F.expr("uuid()"))
    
    # display(result_df)
    
    table_name = f"{get_env_config('golden_consumer_master_database')}.t_derived_consumer_l2"
    print_log(f"merge start table_name: {table_name}")
    merge_condition = get_merge_condition(group_cols)
    merge_t_table(table_name, result_df, merge_condition)
    print_log(f"merge end table_name: {table_name}")
    
    consumer_by_filter_df.unpersist()

In [0]:
task_id = dbutils.widgets.get("task_id")
print_log("task_id: "+task_id)

cid_limit_num = int(get_ex_param("cid_limit_num", "50"))
print(f"cid_limit_num: {cid_limit_num}")

cid_limit_markets = get_ex_param("cid_limit_markets", "AUS,HKG,IDN,JPN,KOR,MYS,NZL,PHL,SGP,THA,TWN,VNM").split(",")
print(f"cid_limit_markets: {cid_limit_markets}")

with StepLogger("t_derived_consumer_l2", "06-2", "consumerlist", task_id=task_id) as logger:

    # 计算的维度
    group_cols = ["consumermdmkey", "scon_brnd_code", "scon_mrkt_code"]

    # 获取consumer数据（已经关联touchpint、ukey排除、source表）
    consumer_join_sources_df = get_consumer_data()

    # 检查是否有数据，没有则直接结束；有数据再执行process_consumer_l2_data
    if consumer_join_sources_df.isEmpty():
        print_log("consumer_join_sources_df 无数据，跳过后续处理")
    else:
        consumer_join_sources_df.cache()
        print_log(f"consumer_join_sources_df count: {consumer_join_sources_df.count()}")
        process_consumer_l2_data(consumer_join_sources_df, task_id, group_cols)

        consumer_join_sources_df.unpersist()